# Session 2. LangGraph: state, nodes, edges

**The loop becomes a shape you declare. Something else runs it.**

- same behaviour; the new owner buys pause, checkpoints, tracing, subgraphs


In [ ]:
import operator
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Reads .env into the process once. Nothing below opens a file again.
load_dotenv()


# Session 1 built the client by hand; this is the same idea one layer up.
# init_chat_model takes a "provider:model" string and returns the model object
# that bind_tools, ToolNode and everything after today expect.
# **kwargs never carries temperature, here or anywhere in this course: current
# Gemini models are documented to loop when it is moved off the default, and a
# loop inside an agent spends a day of quota in one run.
def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. Ten lines, copy them once."""
    # Model ids live in .env as MODEL_CHEAP and MODEL_STRONG. The course calls
    # init_chat_model in exactly one place, config.chat_model, which is why no
    # notebook and no exercise here contains a model name or a key.
    name = os.environ[f"MODEL_{size.upper()}"]
    secret = os.environ["LLM_API_KEY"]
    # Gemini needs its own client class. Over an OpenAI-compatible URL it returns
    # tool calls on turn one and then fails on turn two, because the reasoning
    # signatures attached to an assistant message are not carried across.
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    # Ten lines. Put them in your own repository once and stop thinking about it.
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])


## What the hand-written loop cannot do

**One tool is not a coincidence. It is the ceiling.**

- a second tool: parse a name, branch, grow a dispatch table
- no pause, no resume: the run's state is local variables in a `for`, gone when it returns
- no instrument but `print`: not which node ran, not what the model got on turn three, not how long the tool took


In [ ]:
# The model quotes its argument on some turns and not others, so strip both kinds.
QUOTES = "'" + '"'


# One tool. Roughly the version you wrote: the name is implied, so the return
# value is just the argument.
def parse_action_one(text: str) -> str | None:
    for line in text.splitlines():
        line = line.strip()
        # The entire protocol: a line starting with Action: that has parentheses.
        if line.startswith("Action:") and "(" in line:
            return line[line.index("(") + 1 : line.rindex(")")].strip().strip(QUOTES)
    return None  # no Action line: the model answered instead of calling


# Two tools. The return type became a tuple, so every caller changes with it.
# Every tool you add edits this function, and it is yours, so its bugs are too.
def parse_action_two(text: str) -> tuple[str, str] | None:
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith("Action:") or "(" not in line:
            continue
        body = line[len("Action:") :].strip()
        # Name and argument now have to be pulled apart by hand.
        name = body[: body.index("(")].strip()
        argument = body[body.index("(") + 1 : body.rindex(")")].strip().strip(QUOTES)
        # A whitelist appears the moment there is more than one name to get wrong.
        if name not in ("find_species", "care_notes"):
            return None  # the model invented a tool. Now what?
        return name, argument
    return None


print(parse_action_two('Action: care_notes("mon-01")'))
# camelCase name, so None. Indistinguishable from "there was no tool call".
print(parse_action_two('Action: careNotes("mon-01")'))


## Building the graph: state

**One shared dict. Every node returns a partial update.**

- the update goes to the field's reducer, not to the state


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, StateGraph


class Broken(TypedDict):
    # The schema: a TypedDict, so the keys are fixed and your editor knows them.
    # No reducer on this field, so the default applies, and the default is
    # replacement. It will be overwritten, not appended to.
    messages: list  # no reducer


def add_reply(state) -> dict:
    # Returns one message and never looks at the history.
    return {"messages": [AIMessage("the reply")]}


# The schema goes to the graph, not to the nodes: one shared shape for all of them.
builder = StateGraph(Broken)
builder.add_node("add_reply", add_reply)
builder.add_edge(START, "add_reply")  # START and END are the two built-in nodes
builder.add_edge("add_reply", END)

result = builder.compile().invoke(
    {"messages": [HumanMessage("the question")]},
    # Every invoke in this course passes one. Why, in the loop cell further down.
    config={"recursion_limit": 5},
)
print([m.content for m in result["messages"]])  # count them: one, not two


**One message. The question is gone, nothing warned.**

- you will say "the model keeps forgetting" and look at the model
- history vanishes? look here first. Most common mistake of the session


In [ ]:
from langgraph.graph.message import add_messages


class Fixed(TypedDict):
    # Annotated[type, reducer]. The reducer takes the old list and the update and
    # returns the new list; add_messages appends. This word is the whole fix.
    # It matches on message id: an update carrying an id already in the list
    # replaces that entry instead of duplicating it, which is what stops a
    # replayed history doubling once a checkpointer arrives at session 4.
    messages: Annotated[list, add_messages]


builder = StateGraph(Fixed)
builder.add_node("add_reply", add_reply)  # same node function, untouched
builder.add_edge(START, "add_reply")
builder.add_edge("add_reply", END)

result = builder.compile().invoke(
    {"messages": [HumanMessage("the question")]},
    config={"recursion_limit": 5},
)
print([m.content for m in result["messages"]])  # two messages, in order


In [ ]:
from langgraph.graph import MessagesState

# MessagesState is that one field and nothing else. Import it, never retype it.
print(MessagesState.__annotations__["messages"])


### Nodes and edges

**Node: state in, partial update out. Edge: what runs next.**

- reducers are not a messages feature


In [ ]:
class Note(TypedDict):
    text: str  # no reducer, so every write replaces
    trail: Annotated[list[str], operator.add]  # reducer, so every write concatenates


def shout(state: Note) -> dict:
    # Reads whatever it needs, returns only the keys it changed.
    return {"text": state["text"].upper(), "trail": ["shout"]}


def frame(state: Note) -> dict:
    # Second node, same contract. An ordinary function, nothing registered on it.
    return {"text": f"<<{state['text']}>>", "trail": ["frame"]}


builder = StateGraph(Note)
# add_node registers a plain function under a name; edges refer to that name.
builder.add_node("shout", shout)
builder.add_node("frame", frame)
builder.add_edge(START, "shout")
builder.add_edge("shout", "frame")
builder.add_edge("frame", END)
notes = builder.compile()  # nothing runs before compile, and compile checks the wiring

# draw_mermaid, never draw_mermaid_png: the _png variant posts your graph to
# mermaid.ink and waits for a picture to come back. Text works offline, pastes
# into a markdown file, and is the diagram your defence slides need anyway.
print(notes.get_graph().draw_mermaid())

# text is replaced twice, trail collects both names. Two fields, two reducers.
print(
    notes.invoke(
        # Every key of the schema is supplied up front; trail starts empty.
        {"text": "a graph is a dict and some functions", "trail": []},
        config={"recursion_limit": 5},
    )
)


In [ ]:
# Second silent failure: a node returns a key the schema does not have.
def frame_typo(state: Note) -> dict:
    # One letter. There is no schema key spelled this way.
    return {"text": f"<<{state['text']}>>", "trial": ["frame"]}  # trial, not trail


builder = StateGraph(Note)
builder.add_node("shout", shout)
builder.add_node("frame", frame_typo)  # the only line that differs from before
builder.add_edge(START, "shout")
builder.add_edge("shout", "frame")
builder.add_edge("frame", END)

# trail comes back with one entry and nothing was raised. TypedDict is a hint for
# your type checker, not a runtime check, and the graph writes only the keys it
# knows. A field mysteriously never set: grep it and count the spellings.
print(
    builder.compile().invoke(
        {"text": "one letter", "trail": []},
        config={"recursion_limit": 5},
    )
)


### The model in a node

**Same object session 1 ended on. Now it lives in `messages`.**

- `bind_tools` attaches JSON schemas; their source is session 3


In [ ]:
from langchain_core.tools import tool

# Two dicts standing in for a database: no network, no keys, same result every run.
SPECIES = {"monstera": "mon-01", "ficus": "fic-02", "basil": "bas-03", "cactus": "cac-04"}
CARE = {
    "mon-01": "Water every 7 days. Bright indirect light. Wipe the leaves monthly.",
    "fic-02": "Water every 10 days. Hates being moved. Drops leaves when it sulks.",
    "bas-03": "Water daily. Full sun. Pinch the flowers off to keep the leaves coming.",
    "cac-04": "Water every 21 days in summer, never in winter. Full sun.",
}


# @tool wraps the function; the docstring becomes the description the model reads.
@tool
def find_species(name: str) -> str:
    """Look up the species id of a house plant by its common name."""
    key = name.strip().lower()
    if key not in SPECIES:
        # An error the model can act on: what failed, and what would work instead.
        return f"Unknown plant {name!r}. Known names: {', '.join(sorted(SPECIES))}."
    return SPECIES[key]


# One tool is enough to show a loop. Two DEPENDENT tools are what show why the
# loop has to carry state: nothing can ask for care notes until it knows the id,
# so the id has to survive from one turn into the next, and messages is where.
@tool
def care_notes(species_id: str) -> str:
    """Return watering and light instructions for a species id from find_species."""
    # The docstring names find_species on purpose: it tells the model the order.
    if species_id not in CARE:
        # Where a guessed id lands. The model reads this and recovers next turn;
        # writing return values like this one is most of session 3.
        return f"No care notes for {species_id!r}. Call find_species first to get a valid id."
    return CARE[species_id]


TOOLS = [find_species, care_notes]
print([t.name for t in TOOLS])  # the tool name is the function name, and the model sees it


In [ ]:
QUESTION = "My monstera looks sad. How often should I be watering it?"

model = chat_model("strong")
# Returns a NEW model object with the schemas attached to every request it makes.
# Nothing is sent here.
bound = model.bind_tools(TOOLS)

# One request, one turn. A plain dict is accepted in place of a HumanMessage.
ai = bound.invoke([{"role": "user", "content": QUESTION}])
print("type:      ", type(ai).__name__)
print("content:   ", repr(ai.content))  # usually empty on a tool-calling turn
# name, args, id. The id is load-bearing: the tool result has to come back
# carrying it, or the provider cannot match a result to the call that asked.
print("tool_calls:", ai.tool_calls)


**The model asked. Nothing has run.**


### The conditional edge, by hand

**A function from state to the name of the next node.**

- six lines, no idea beyond that sentence


In [ ]:
from langgraph.prebuilt import ToolNode


def call_model(state: MessagesState) -> dict:
    # Return only what changed. The reducer appends it. Returning the whole list
    # here would append a copy of the history to the history.
    # What goes in is the reply object exactly as it came back, never a rebuild
    # of it. That is a rule, not brevity, and the last section says why.
    return {"messages": [bound.invoke(state["messages"])]}


def should_continue(state: MessagesState) -> str:
    last = state["messages"][-1]
    # The whole routing decision is one attribute on the last message.
    if last.tool_calls:
        return "tools"  # a node name, as a string
    return END  # END is a legal destination too, and stops the run


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
# One node holds every tool and picks by name at run time. Next section: how.
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
# The third argument lists the possible destinations. It constrains the drawing,
# not the routing: leave it out and the diagram shows every node as reachable.
builder.add_conditional_edges("model", should_continue, ["tools", END])
builder.add_edge("tools", "model")  # this edge, and only this edge, is the loop
manual = builder.compile()

print(manual.get_graph().draw_mermaid())

# recursion_limit is not paranoia. The default in langgraph 1.2 is 10007, not the
# 25 everyone assumes, so a loop that never terminates will call a paid endpoint
# for a very long time before anything stops it. Every invoke in this course
# passes a small one.
result = manual.invoke(
    # A plain dict again: add_messages coerces it into a HumanMessage.
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8},
)
# Length of the history, then the last message. Both read straight out of state.
print(len(result["messages"]), "messages")
print(result["messages"][-1].content)


**Those six lines ship prebuilt, as `tools_condition`.**

- buys: two lines instead of eight
- costs: read the comment in the next cell first


In [ ]:
# This symbol returns zero hits across the current documentation, while ToolNode
# is on twelve pages, and the canonical loop written there uses a hand-rolled
# should_continue. Its own docstring ships two import lines that raise
# ImportError on this stack. A student sent to the docs finds nothing.
from langgraph.prebuilt import tools_condition

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
# Its return type is Literal["tools", "__end__"], so this name is not a choice.
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
# No should_continue and no destination list. Same routing as the eight lines above.
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")
graph = builder.compile()  # the graph the rest of the notebook runs

print(graph.get_graph().draw_mermaid())  # same diagram as the hand-written version


In [ ]:
# The same graph with the tool node called anything else.
builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("plant_tools", ToolNode(TOOLS))  # the only change in the cell
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("plant_tools", "model")  # the loop edge is fine; the router is not

try:
    # tools_condition can still return "tools", and no node answers to that name.
    builder.compile()
except ValueError as error:
    # A compile-time error, so at least you find it now rather than at run time.
    print("ValueError:", error)


### ToolNode and the whole loop

**`ToolNode` is the session-1 block you wrote by hand.**

- reads `tool_calls`, runs each, returns a `ToolMessage` per call


In [ ]:
# Same question, the prebuilt graph. The output below is the point of today.
result = graph.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8},
)

# pretty_print shows the role, the tool call, and the tool_call_id each
# ToolMessage answers. Read them in order: this is the entire run.
for message in result["messages"]:
    message.pretty_print()


**Six messages. The loop ran twice and you wrote no loop.**

- 1 human; 2 AI `find_species("monstera")`; 3 tool `mon-01`
- 4 AI `care_notes("mon-01")`: that argument came from message 3
- 5 tool notes; 6 AI answer, no calls, routed to `END`
- the id pairing 2 with 3 is what you will read in every trace from now on


In [ ]:
# Two things go wrong inside ToolNode, and they are not symmetric. Both cells
# use a fake model node, so they cost nothing and behave the same every time.
def hallucinate(state: MessagesState) -> dict:
    # Builds the AIMessage itself, so no request goes out.
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "findSpecies",  # no such tool
                        "args": {"name": "monstera"},
                        "id": "call-1",  # invented here; normally the provider sets it
                    }
                ],
            )
        ]
    }


builder = StateGraph(MessagesState)
builder.add_node("model", hallucinate)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
builder.add_edge("model", "tools")  # plain edge: no routing, straight into the tools
builder.add_edge("tools", END)

out = builder.compile().invoke({"messages": []}, config={"recursion_limit": 5})
# A ToolMessage, not an exception, and it lists the names that would have worked.
# The model reads that and calls the right one: the loop repairs itself.
print(out["messages"][-1].content)


In [ ]:
from langchain_core.tools import tool


@tool
def flaky(species_id: str) -> str:
    """Looks like any other tool right up until the day it does not."""
    # Stands in for the HTTP call that every real tool eventually makes.
    raise TimeoutError("upstream took too long")


def call_flaky(state: MessagesState) -> dict:
    # Same fake-model trick, one call, aimed at the tool that throws.
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {"name": "flaky", "args": {"species_id": "mon-01"}, "id": "call-1"}
                ],
            )
        ]
    }


builder = StateGraph(MessagesState)
builder.add_node("model", call_flaky)
builder.add_node("tools", ToolNode([flaky]))  # only the throwing tool is registered
builder.add_edge(START, "model")
builder.add_edge("model", "tools")
builder.add_edge("tools", END)

try:
    builder.compile().invoke({"messages": []}, config={"recursion_limit": 5})
except TimeoutError as error:
    # ToolNode re-raises anything that is not its own invocation error, so one
    # HTTP timeout inside your tool ends the run and takes the conversation with
    # it. Your tool catches its own exceptions and returns a string saying what
    # went wrong and what to try instead.
    print("the run is over:", type(error).__name__, error)


**A wrong tool name is handled. An exception is not.**


## init_chat_model

**Same `.env`, same `MODEL_CHEAP` and `MODEL_STRONG`. New return value.**

- a model object with `bind_tools`, not a client you post dicts to


In [ ]:
cheap = chat_model("cheap")
strong = chat_model("strong")
# Same class both times: the size only picks a different id out of .env.
print(type(cheap).__name__, "|", type(strong).__name__)

# The reply object from the model node, from earlier in this notebook. All three
# lines print what a hand-built AIMessage(content="", tool_calls=[...]) throws away.
print("id:               ", ai.id)  # set by the provider, not by us
print("additional_kwargs:", list(ai.additional_kwargs))  # provider-specific extras
print("response_metadata:", list(ai.response_metadata))  # finish reason, token counts


**Append the model's reply object itself. Never rebuild it.**

- providers hang reasoning signatures, cache markers and ids on it
- a rebuild prints identically, then fails on the next turn


## Practice

**Port your assistant onto a graph, in your own repository.**

1. `MessagesState`, model node, `ToolNode`, one conditional edge
2. `should_continue` by hand, then swap in `tools_condition`
3. session-1 tool becomes `@tool` with a real docstring: that text is what the model reads
4. delete `parse_action` and the format prompt. Delete, not comment out


**Independent tools never make the loop come back. Nothing gets tested.**

5. a second and a third tool, one taking another's output: a lookup returning an id, and something that takes that id
6. every `invoke` passes a small `recursion_limit`
7. commit `runs/session-02.md`: the diagram, and six-plus messages whose fourth cites the third
8. ask early: today's three silent failures look like a stupid model

The notebook is a reference, not a source: your domain differs and the code will not copy across.


## Observability

**`print` stops paying for itself around the third node.**

- Langfuse in Docker on your own machine: no signup, no region, no quota
- a trace per run, a span per node, the real message list, how long each step took
- the traces never leave your laptop


Next to your `.env`:

```
curl -sSLO https://raw.githubusercontent.com/langfuse/langfuse/v3.224.0/docker-compose.yml
```

Then `docker-compose.override.yml` beside it. Compose applies it automatically, no `-f`.

```yaml
services:
  langfuse-web:
    environment:
      LANGFUSE_INIT_ORG_ID: agents-course
      LANGFUSE_INIT_ORG_NAME: LLM Agents Course
      LANGFUSE_INIT_PROJECT_ID: assistant
      LANGFUSE_INIT_PROJECT_NAME: My assistant
      LANGFUSE_INIT_PROJECT_PUBLIC_KEY: ${LANGFUSE_PUBLIC_KEY}
      LANGFUSE_INIT_PROJECT_SECRET_KEY: ${LANGFUSE_SECRET_KEY}
      LANGFUSE_INIT_USER_EMAIL: ${LANGFUSE_USER_EMAIL}
      LANGFUSE_INIT_USER_NAME: Student
      LANGFUSE_INIT_USER_PASSWORD: ${LANGFUSE_USER_PASSWORD}
```

- four new lines in `.env`: `LANGFUSE_HOST=http://localhost:3000`, plus three strings you invent for the two keys and your login
- keep the `pk-lf-` and `sk-lf-` prefixes
- never quote a `LANGFUSE_INIT_*` value: quoted, it is read literally, init silently does nothing, and you get an empty Langfuse whose keys are rejected

```
docker compose up -d
```

Two or three minutes the first time, seconds after that.

- compose and your code read the same `.env`: both sides agree by construction


In [ ]:
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

# Reads LANGFUSE_HOST and both keys straight out of the environment.
client = get_client()
# The host, not the keys. A key never gets printed into a file you commit.
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", client.auth_check())

handler = CallbackHandler()  # a fresh handler per run gives one trace per run
traced = graph.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    # The only change at the call site: the handler rides in the same config dict
    # as recursion_limit. Nothing inside the graph changed.
    config={"recursion_limit": 8, "callbacks": [handler]},
)

# The SDK batches and sends in the background. A script exits and flushes on the
# way out; a notebook kernel never exits, so the events sit in the buffer and the
# trace never appears. This is the single most common reason a student's trace is
# missing. Still nothing on the second run: the keys in .env and the keys the
# container was created with have drifted apart, and check_env.py will say so.
client.flush()

print(traced["messages"][-1].content)


**Find three things in the trace and name each out loud.**

- the `tools` span, with both tool results inside it
- the second model call: history appended, not replaced
- the token counts and the per-step latency, session 3


## Discussion

**You built the same agent twice and the second is longer.**

- where does the graph pay, and where is the `for` loop right?
- what separates them: tool count, pausing, another reader?
- argue for the loop: one call, no branching, a straight line


---

## Not taught today

**A conditional edge does not have to route to a tool.**

- it routes on anything computable from state, a classifier included
- `should_continue` with more return values, not a new pattern


In [ ]:
# Read-only. Nothing here is used anywhere else in this notebook. Session 9 puts
# it beside four other orchestration shapes and the criteria for choosing, which
# is the hard part. Read it now if you want, do not build on it yet.
class Routed(TypedDict):
    text: str
    kind: str  # written by classify, read by route
    reply: str


def classify(state: Routed) -> dict:
    text = state["text"].lower()
    if text.endswith("?"):
        return {"kind": "question"}
    if text.startswith(("do ", "make ", "send ")):
        return {"kind": "command"}
    return {"kind": "other"}


def route(state: Routed) -> str:
    # Same signature as should_continue: state in, name of the next node out.
    return state["kind"]


# builder.add_conditional_edges("classify", route, ["question", "command", "other"])
#
# Three destinations instead of two. A classifier written with a model in it
# replaces the if-statements above and changes nothing about the wiring.
